In [1]:
# !python -m spacy download el_core_news_sm

In [2]:
# pip install spacy

In [2]:
import spacy
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
dfp = pd.read_csv("poems_sentiment.csv")
dfr = pd.read_csv("sentiment.csv")
dfr = dfr.drop(columns = ['album', 'annotation_count', 'featured_artists' ,'lyrics', 'date', 'pageviews', 'num_intros', 'num_verses', 'num_choruses', 
'num_pre_chorus', 'num_bridges', 'num_outros', 'num_instrumentals', 'line_count', 'word_count', 'unique_word_count', 'lexical_diversity'])
dfr = dfr.rename(columns = {'clean_lyrics': 'lyrics'})
dfp = dfp.rename(columns = {'Poet' : 'artist', 'Title': 'title', 'Lyrics': 'lyrics'})
dfp = dfp.drop(columns = ['Author_ID', 'Poem_ID'])
dfp['genre'] = 'poem'
dfr['genre'] = 'rap'
dfr = dfr[dfr['artist'] != 'Kanonenfieber']
df = pd.concat([dfr, dfp])

In [5]:
# 1. Φόρτωση του Ελληνικού Μοντέλου Γραμματικής
nlp = spacy.load("el_core_news_sm")

def get_pos_percentages(text):
    """
    Διαβάζει ένα κείμενο, μετράει τα μέρη του λόγου 
    και επιστρέφει το ποσοστό τους (%) επί του συνόλου των λέξεων.
    """
    if not isinstance(text, str) or len(text.strip()) == 0:
        return {"VERB_pct": 0, "NOUN_pct": 0, "ADJ_pct": 0}
    
    # Το spaCy αναλύει γραμματικά το κείμενο
    doc = nlp(text)
    
    counts = {"VERB": 0, "NOUN": 0, "ADJ": 0}
    total_valid_words = 0
    
    for token in doc:
        # Αγνοούμε σημεία στίξης και κενά για να είναι δίκαιη η μέτρηση
        if not token.is_punct and not token.is_space:
            total_valid_words += 1
            if token.pos_ in counts:
                counts[token.pos_] += 1
                
    # Αν το κείμενο είναι άδειο
    if total_valid_words == 0:
        return {"VERB_pct": 0, "NOUN_pct": 0, "ADJ_pct": 0}
        
    # Υπολογισμός ποσοστών %
    return {
        "VERB_pct": (counts["VERB"] / total_valid_words) * 100,
        "NOUN_pct": (counts["NOUN"] / total_valid_words) * 100,
        "ADJ_pct": (counts["ADJ"] / total_valid_words) * 100
    }

In [ ]:
rap = dfr['lyrics'].apply(get_pos_percentages).apply(pd.Series)
rap.to_csv("rap_pos.csv", index=False)

In [ ]:
poem = dfp['lyrics'].apply(get_pos_percentages).apply(pd.Series)
poem.to_csv("poem_pos.csv", index=False)

In [ ]:
dfr = pd.read_csv("rap_pos.csv")
dfp = pd.read_csv("poem_pos.csv")
dfr = dfr.drop(columns = ["Unnamed: 0"])
dfp = dfp.rename(columns = {'Poet' : 'artist', 'Title': 'title', 'Lyrics': 'lyrics'})
dfp = dfp.drop(columns = ["Unnamed: 0"])
dfp['genre'] = 'poem'
dfr['genre'] = 'rap'
dfr = dfr[dfr['artist'] != 'Kanonenfieber']
df = pd.concat([dfr, dfp])

# Υπολογισμός μέσων όρων (σε ποσοστά %) ανά είδος
pos_means = df.groupby('genre')[['VERB_pct', 'NOUN_pct', 'ADJ_pct', 'ADV_pct', 'PRON_pct']].mean()
print(pos_means)